In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
def predict(X, weights, lambdaa = 1):
    net = np.dot(X, weights)
    return (1 / (1 + np.exp(-lambdaa * net)))

def calculate_dw(X, y, y_pred):
    return ((y_pred - y) * (y_pred) * (1 - y_pred) * X)

def calculate_error(yi, y_pred):
    return np.square(y_pred - yi)

In [13]:
def train_vanilla(X, y, weights, epochs=500, lr=0.001, epislon=1e-8, beta=0.5):
    vel = 0
    for epoch in range(epochs):
        error = 0
        dw = 0
        for Xi, yi in zip(X, y):
            y_pred = predict(Xi, weights)
            dw += calculate_dw(Xi, yi, y_pred)
            error += calculate_error(yi, y_pred)

        vel = (beta * vel) + ((1 - beta) * (dw**2))
        weights -= ((lr * dw) / (np.sqrt(vel + epislon)))
        error /= (2 * len(X))

        if (epoch+1) % 50 == 0:
            print(f'Weights after epoch {epoch+1} : ',weights)
            print(f'Error after epoch {epoch+1} : ',error)

In [14]:
def train_stochastic(X, y, weights, epochs=500, lr=0.001, epislon=1e-8, beta=0.5):
    vel = 0
    for epoch in range(epochs):
        error = 0
        for Xi, yi in zip(X, y):
            y_pred = predict(Xi, weights)
            dw = calculate_dw(Xi, yi, y_pred)
            error += calculate_error(yi, y_pred)
            vel = (beta * vel) + ((1 - beta) * (dw**2))
            weights -= ((lr * dw) / (np.sqrt(vel + epislon)))

        error /= (2 * len(X))

        if (epoch+1) % 50 == 0:
            print(f'Weights after epoch {epoch+1} : ',weights)
            print(f'Error after epoch {epoch+1} : ',error)

In [19]:
def train_mini_batch(X, y, weights, epochs=500, lr=0.001, epislon=1e-8, beta=0.5, bs=32):
    vel = 0
    for epoch in range(epochs):
        error = 0
        dw = 0
        i = 0
        for Xi, yi in zip(X, y):
            y_pred = predict(Xi, weights)
            dw += calculate_dw(Xi, yi, y_pred)
            i += 1
            error += calculate_error(yi, y_pred)

            if i%bs == 0 or i == len(X):
                vel = (beta * vel) + ((1 - beta) * (dw**2))
                weights -= ((lr * dw) / (np.sqrt(vel + epislon)))
                dw = 0

        error /= (2 * len(X))

        if (epoch+1) % 50 == 0:
            print(f'Weights after epoch {epoch+1} : ',weights)
            print(f'Error after epoch {epoch+1} : ',error)

In [7]:
df = pd.read_csv('bank_note.csv')
df.insert(4,'x0',1)

In [8]:
X = df[['variance','skewness','curtosis','entropy','x0']].values
y = df['class']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=13, test_size=0.2)
weights = input('Enter 4 weights and 1 bias : ').split()
weights = np.array([float(weight) for weight in weights], dtype='longdouble')

In [16]:
print('RMSPROP (Vanilla)')
train_vanilla(X_train,y_train,weights.copy())

RMSPROP (Vanilla)
Weights after epoch 50 :  [ 0.4493069  -0.25037206  0.74903664 -0.35157574  0.06198668]
Error after epoch 50 :  0.2163499406573366
Weights after epoch 100 :  [ 0.39930191 -0.29993683  0.69883838 -0.40205345  0.11442909]
Error after epoch 100 :  0.20305518777499387
Weights after epoch 150 :  [ 0.34930281 -0.34924393  0.64868769 -0.45234997  0.16504475]
Error after epoch 150 :  0.1900083577107779
Weights after epoch 200 :  [ 0.29931734 -0.39733326  0.59856894 -0.50253783  0.21538433]
Error after epoch 200 :  0.1772488916715075
Weights after epoch 250 :  [ 0.24933116 -0.40912908  0.54843979 -0.55255459  0.26544664]
Error after epoch 250 :  0.16454668072104164
Weights after epoch 300 :  [ 0.19938231 -0.42414932  0.49835217 -0.60246006  0.31540519]
Error after epoch 300 :  0.1517611330275018
Weights after epoch 350 :  [ 0.14946333 -0.4461929   0.44831095 -0.65229241  0.36532279]
Error after epoch 350 :  0.13943686248340303
Weights after epoch 400 :  [ 0.09956499 -0.4727654

In [17]:
print('RMSProp (Stochastic)')
train_stochastic(X_train,y_train,weights.copy())

RMSProp (Stochastic)
Weights after epoch 50 :  [-2.86730008 -1.27913962 -1.75971618  0.38487542  4.40293968]
Error after epoch 50 :  0.005154824874154497
Weights after epoch 100 :  [-3.76696051 -1.72619747 -2.38285566  0.49903036  5.67723716]
Error after epoch 100 :  0.005210018111249248
Weights after epoch 150 :  [-4.5312659  -2.08855458 -2.85993662  0.54560109  6.48693452]
Error after epoch 150 :  0.005040569176895971
Weights after epoch 200 :  [-5.22718951 -2.41013561 -3.28224936  0.5766594   7.12537978]
Error after epoch 200 :  0.004839186751381252
Weights after epoch 250 :  [-5.93101704 -2.67360922 -3.66909961  0.62812689  7.66466291]
Error after epoch 250 :  0.004611839250624377
Weights after epoch 300 :  [-6.64252837 -2.90169328 -4.0308283   0.71166195  8.15551326]
Error after epoch 300 :  0.004404283069751225
Weights after epoch 350 :  [-7.34808213 -3.10411553 -4.37652138  0.80717939  8.61703642]
Error after epoch 350 :  0.0042898737027841525
Weights after epoch 400 :  [-8.0168

In [20]:
print('RMSProp (Mini Batch)')
train_mini_batch(X_train,y_train,weights.copy())

RMSProp (Mini Batch)
Weights after epoch 50 :  [-0.90922785 -0.4857604  -0.5070907  -0.2269912   0.63426035]
Error after epoch 50 :  0.013924555020031978
Weights after epoch 100 :  [-1.33849478 -0.70436909 -0.83546024 -0.0556999   1.48323007]
Error after epoch 100 :  0.006653368091379175
Weights after epoch 150 :  [-1.61703277e+00 -8.45753275e-01 -1.02957262e+00 -1.39539789e-03
  1.88733160e+00]
Error after epoch 150 :  0.005311757525720115
Weights after epoch 200 :  [-1.83189912 -0.94747119 -1.1682011   0.01471029  2.14003183]
Error after epoch 200 :  0.004784596523900017
Weights after epoch 250 :  [-2.01182614 -1.03352499 -1.28306475  0.01672463  2.3262747 ]
Error after epoch 250 :  0.00448527569604139
Weights after epoch 300 :  [-2.17085422 -1.10825877 -1.38315843  0.01551631  2.47755462]
Error after epoch 300 :  0.004287034729684852
Weights after epoch 350 :  [-2.31613325 -1.17447407 -1.47309208  0.01441242  2.60858229]
Error after epoch 350 :  0.004143551415122579
Weights after ep